# Get Sample Data

In [ ]:
import os
import random
import pandas as pd
from sklearn.model_selection import train_test_split

def sample_txt_files_with_subdir(base_dir, name, output_excel):

    file_name = name
    file_path = os.path.join(base_dir, file_name)

    data = pd.read_excel(file_path)

    data = data.dropna(subset='Option 3')
    print(data.shape[0])

    data['Option 4'] = data['Option 4'].fillna('لاشيء مما سبق')

    remove, sample = train_test_split(data,
                                      test_size = 200,
                                      random_state = 42)

    print(sample.shape)

    output_path = os.path.join(base_dir, output_excel)
    sample.to_excel(output_path, index=False)
    print(f"Saved {len(sample)} records to {output_excel}")

directory = 'Math (Primary School)'
sample_txt_files_with_subdir(directory, 'test.xlsx', 'sample_math.xlsx')

346
(200, 15)
Saved 200 records to sample_math.xlsx


# English Prompts

## Zero Shot

In [ ]:
math = pd.read_excel('sample_math.xlsx')

prompt = '''Your task is to select and write only the correct answer from the options (a, b, c, d).
To answer, first write the number of the option (a, b, c, d) and then write the answer. Strictly follow the following format for the option:
a/b/c/d) Answer

The question you have to answer:
Question:
'''

zero_Shot = []
for i, ques in enumerate(math['Question']):
    zero_Shot.append(prompt + ques + '\n Options:\na) ' + str(math['Option 1'].iloc[i]) + '\nb) ' + str(math['Option 2'].iloc[i]) + '\nc) ' + str(math['Option 3'].iloc[i]) + '\nd) ' + str(math['Option 4'].iloc[i]) + '\n\nAnswer:')

zs_math = pd.DataFrame()
zs_math['prompt'] = zero_Shot
zs_math.to_excel('Math-QA-EnglishPrompt-Zero Shot.xlsx', index = False)

## Few Shot

In [ ]:
prompt_fs = '''Your task is to select and write only the correct answer from the options (a, b, c, d).
To answer, first write the number of the option (a, b, c, d) and then write the answer. Strictly follow the following format for the option:
a/b/c/d) Answer

Example 1
Question:
المقاومة الكهربائية لمصباح مكتوب عليه 220 فولت ، 100 واط  هي

Options:
a) 220 أوم
b) 202 أوم
c) 100 أوم
d) 484 أوم

Answer:
d) 484 أوم

Example 2
Question:
يتشابه عملها مع عمل كرات الدم البيضاء فى الإنسان

Options:
a) جلوكوزيدات
b) مستقبلات
c) أحماض أمينية غير بروتينية
d) سيفالوسبورين

Your answer:
b) مستقبلات

Example 3
Question:
ماذا يسمى العدد 50 في عملية الطرح : 70 – 20 = 50 ؟

Options:
a) ناتج الطرح
b) المطروح
c) المطروح منه
d) لاشيء مما سبق

Your answer:
a) ناتج الطرح

The question you have to answer:
Question:
'''

few_Shot = []
for i, ques in enumerate(math['Question']):
    few_Shot.append(prompt_fs + ques + '\n Options:\na) ' + str(math['Option 1'].iloc[i]) + '\nb) ' + str(math['Option 2'].iloc[i]) + '\nc) ' + str(math['Option 3'].iloc[i]) + '\nd) ' + str(math['Option 4'].iloc[i]) + '\n\nAnswer:')

fs_math = pd.DataFrame()
fs_math['prompt'] = few_Shot
fs_math.to_excel('Math-QA-EnglishPrompt-Few Shot.xlsx', index = False)

## CoT

In [ ]:
prompt_cot = '''Your task is to select and write only the correct answer from the options (a, b, c, d).
To answer, first write the number of the option (a, b, c, d) and then write the answer. Strictly follow the following format for the option:
a/b/c/d) Answer

Follow these reasoning steps to determine the correct answer:

Step 1: Understand the question
Carefully read and comprehend the question. Identify what is being asked, including the underlying need and the context.

Step 2: Analyze the options
Carefully read each of the four options. Consider the meaning of each option and how it relates to the question.

Step 3: Evaluate the options
Apply your knowledge and reasoning to assess each option. Eliminate clearly incorrect options. Compare the remaining options to determine which is most accurate.

Step 4: Select the best answer
Based on your evaluation in the previous steps, choose the single most logical and correct option.

Step 5: Output the answer
Write only the correct answer in this exact format:
a/b/c/d) Answer

Example 1
Question:
المقاومة الكهربائية لمصباح مكتوب عليه 220 فولت ، 100 واط  هي

Options:
a) 220 أوم
b) 202 أوم
c) 100 أوم
d) 484 أوم

Thinking:
- Step 1: Understand the question
We are asked to calculate the electrical resistance of a lamp rated at 220 volts and 100 watts.
- Step 2: Analyze the options
The resistance must be computed based on the given voltage and power. The options are 220 ohms, 202 ohms, 100 ohms, and 484 ohms.
- Step 3: Evaluate the options
We know the formula for electrical resistance in terms of voltage and power is:
R = V^2/P
Substitute the given values:
R = 220^2/100 = 48400/100 = 484 ohms
- Step 4: Select the correct answer
From the options, 484 أوم corresponds to option d.

Final Answer:
d) 484 أوم

Example 2
Question:
يتشابه عملها مع عمل كرات الدم البيضاء فى الإنسان

Options:
a) جلوكوزيدات
b) مستقبلات
c) أحماض أمينية غير بروتينية
d) سيفالوسبورين

Thinking:
- Step 1: Understand the question
We are asked: Which of these functions similarly to white blood cells in humans?
White blood cells (WBCs) are part of the immune system — they fight infections by detecting and neutralizing pathogens (bacteria, viruses, etc.).
- Step 2: Analyze the options
a) Glycosides (جلوكوزيدات): These are sugar-based compounds, not immune-related.
b) Receptors (مستقبلات): Receptors help in detecting and recognizing substances (including pathogens), playing a role in immunity and signaling.
c) Non-protein amino acids (أحماض أمينية غير بروتينية): These are rare metabolic intermediates, not directly involved in immune function.
d) Cephalosporins (سيفالوسبورين): These are antibiotics; they kill bacteria but are not a cellular component of the immune system.
- Step 3: Evaluate the options
Which of these functions similarly to WBCs — that is, by detecting or responding to pathogens?
Receptors help the immune system recognize invaders — similar to how WBCs detect and respond to pathogens.
- Step 4: Select the correct answer
The option that most closely mimics the immune detection function of WBCs is receptors (مستقبلات).

Final Answer:
b) مستقبلات

Example 3
Question:
ماذا يسمى العدد 50 في عملية الطرح : 70 – 20 = 50 ؟

Options:
a) ناتج الطرح
b) المطروح
c) المطروح منه
d) لاشيء مما سبق

Thinking:
- Step 1: Understand the question
We are asked: What is the name of the number 50 in the subtraction operation 70 – 20 = 50?
In subtraction:
The first number (70) is called the minuend (المطروح منه).
The second number (20) is called the subtrahend (المطروح).
The result (50) is called the difference (ناتج الطرح).
- Step 2: Analyze the options
a) ناتج الطرح: "Result of subtraction" — this is the correct mathematical term for the outcome of the subtraction (difference).
b) المطروح: "The number being subtracted" — this is 20, not 50.
c) المطروح منه: "The number from which another number is subtracted" — this is 70, not 50.
d) لاشيء مما سبق: "None of the above" — not applicable since option (a) is correct.
- Step 3: Evaluate the options
The number 50 is the result of the subtraction, also known as the difference.
- Step 4: Select the correct answer
The correct answer is ناتج الطرح.

Final Answer:
a) ناتج الطرح

The question you have to answer:
Question:
'''

CoT = []
for i, ques in enumerate(math['Question']):
    CoT.append(prompt_cot + ques + '\n Options:\na) ' + str(math['Option 1'].iloc[i]) + '\nb) ' + str(math['Option 2'].iloc[i]) + '\nc) ' + str(math['Option 3'].iloc[i]) + '\nd) ' + str(math['Option 4'].iloc[i]))

cot_math = pd.DataFrame()
cot_math['prompt'] = CoT
cot_math.to_excel('Math-QA-EnglishPrompt-CoT.xlsx', index = False)

# Arabic Prompts

## Zero Shot

In [ ]:
math = pd.read_excel('sample_math.xlsx')

prompt = '''لديك سؤال وأربع إجابات بديلة. مهمتك هي تحديد الإجابة الصحيحة وكتابتها فقط من بين الخيارات (أ، ب، ج، د).
للإجابة، اكتب أولاً رقم الخيار (أ، ب، ج، د) ثم اكتب الإجابة. اتبع التنسيق التالي بدقة للخيار:
أ / ب / ج / د) الإجابة

السؤال الذي يجب عليك الإجابة عليه:
السؤال:

'''

zero_Shot = []
for i, ques in enumerate(math['Question']):
    zero_Shot.append(prompt + ques + '\n الخيارات:\nأ) ' + str(math['Option 1'].iloc[i]) + '\nب) ' + str(math['Option 2'].iloc[i]) + '\nج) ' + str(math['Option 3'].iloc[i]) + '\nد) ' + str(math['Option 4'].iloc[i]) + '\n\nالإجابة:')

zs_math = pd.DataFrame()
zs_math['prompt'] = zero_Shot
zs_math.to_excel('Math-QA-ArabicPrompt-Zero Shot.xlsx', index = False)

## Few Shot

In [ ]:
prompt_fs = '''لديك سؤال وأربع إجابات بديلة. مهمتك هي تحديد الإجابة الصحيحة وكتابتها فقط من بين الخيارات (أ، ب، ج، د).
للإجابة، اكتب أولاً رقم الخيار (أ، ب، ج، د) ثم اكتب الإجابة. اتبع التنسيق التالي بدقة للخيار:
أ / ب / ج / د) الإجابة

مثال 1:
السؤال:
المقاومة الكهربائية لمصباح مكتوب عليه 220 فولت ، 100 واط  هي

الخيارات:
أ) 220 أوم
ب) 202 أوم
ج) 100 أوم
د) 484 أوم

الإجابة:
د) 484 أوم

مثال 2:
السؤال:
يتشابه عملها مع عمل كرات الدم البيضاء فى الإنسان

الخيارات:
أ) جلوكوزيدات
ب) مستقبلات
ج) أحماض أمينية غير بروتينية
د) سيفالوسبورين

الإجابة:
ب) مستقبلات

مثال 3:
السؤال:
ماذا يسمى العدد 50 في عملية الطرح : 70 – 20 = 50 ؟

الخيارات:
أ) ناتج الطرح
ب) المطروح
ج) المطروح منه
د) لاشيء مما سبق

الإجابة:
أ) ناتج الطرح

السؤال الذي يجب عليك الإجابة عليه:
السؤال:
'''

few_Shot = []
for i, ques in enumerate(math['Question']):
    few_Shot.append(prompt_fs + ques + '\n الخيارات:\nأ) ' + str(math['Option 1'].iloc[i]) + '\nب) ' + str(math['Option 2'].iloc[i]) + '\nج) ' + str(math['Option 3'].iloc[i]) + '\nد) ' + str(math['Option 4'].iloc[i]) + '\n\nالإجابة:')

fs_math = pd.DataFrame()
fs_math['prompt'] = few_Shot
fs_math.to_excel('Math-QA-ArabicPrompt-Few Shot.xlsx', index = False)

## CoT

In [ ]:
prompt_cot = '''لديك سؤال وأربع إجابات بديلة. مهمتك هي تحديد الإجابة الصحيحة وكتابتها فقط من بين الخيارات (أ، ب، ج، د).
للإجابة، اكتب أولاً رقم الخيار (أ، ب، ج، د) ثم اكتب الإجابة. اتبع التنسيق التالي بدقة للخيار:
أ / ب / ج / د) الإجابة

اتبع الخطوات التالية للوصول إلى الإجابة الصحيحة:

الخطوة 1: فهم السؤال
اقرأ السؤال بعناية لفهم ما يُطلب منك تحديداً. حاول فهم الحاجة والسياق الكامن خلف السؤال.

الخطوة 2: تحليل الخيارات
اقرأ كل خيار بعناية. فكّر في معنى كل خيار وكيف يرتبط بالسؤال.

الخطوة 3: تقييم الخيارات
استخدم معرفتك ومعلوماتك ذات الصلة لتقييم كل خيار. استبعد الخيارات الخاطئة بشكل واضح، ثم قارن بين الخيارات المتبقية لتحديد الأنسب.

الخطوة 4: اختيار أفضل إجابة
اختر الخيار الأكثر منطقية وصحة بناءً على التقييم في الخطوات السابقة.

الخطوة 5: كتابة الإجابة
اكتب الإجابة الصحيحة فقط باستخدام هذا التنسيق الصارم:
أ / ب / ج / د) الإجابة

مثال 1:
السؤال:
المقاومة الكهربائية لمصباح مكتوب عليه 220 فولت ، 100 واط  هي

الخيارات:
أ) 220 أوم
ب) 202 أوم
ج) 100 أوم
د) 484 أوم

التحليل:
- الخطوة 1: فهم السؤال
المطلوب حساب المقاومة الكهربائية لمصباح قدرته 100 واط وجهده 220 فولت.
- الخطوة 2: تحليل الخيارات
لحساب المقاومة نستخدم الجهد والقدرة الكهربائية المعطاة. الخيارات هي 220 أوم، 202 أوم، 100 أوم، و484 أوم.
- الخطوة 3: تقييم الخيارات
نستخدم العلاقة التالية لحساب المقاومة:
R = V^2/P
نعوض القيم المعطاة:
R = 220^2/100 = 48400/100 = 484 أوم
- الخطوة 4: اختيار الإجابة الصحيحة
من بين الخيارات، 484 أوم هي الإجابة الصحيحة، وهي الخيار د.

الإجابة النهائية:
د) 484 أوم

مثال 2:
السؤال:
يتشابه عملها مع عمل كرات الدم البيضاء فى الإنسان

الخيارات:
أ) جلوكوزيدات
ب) مستقبلات
ج) أحماض أمينية غير بروتينية
د) سيفالوسبورين

التحليل:
- الخطوة 1: فهم السؤال
السؤال يطلب: أي من الخيارات التالية يتشابه عمله مع عمل كرات الدم البيضاء في جسم الإنسان؟
كرات الدم البيضاء هي جزء من جهاز المناعة؛ وظيفتها الأساسية التعرف على مسببات الأمراض (بكتيريا، فيروسات، إلخ) ومهاجمتها.
- الخطوة 2: تحليل الخيارات
أ) جلوكوزيدات: مركبات سكرية؛ ليست جزءاً من جهاز المناعة.
ب) مستقبلات: المستقبلات تلعب دوراً مهماً في التعرف على مسببات الأمراض وإرسال إشارات لتحفيز الاستجابة المناعية.
ج) أحماض أمينية غير بروتينية: مركبات نادرة؛ ليست مرتبطة مباشرة بوظيفة جهاز المناعة.
د) سيفالوسبورين: مضاد حيوي يقضي على البكتيريا لكنه ليس مكوناً خلوياً في جهاز المناعة.
- الخطوة 3: تقييم الخيارات
ما الذي يشبه وظيفة كرات الدم البيضاء — أي القدرة على التعرف على مسببات الأمراض والاستجابة لها؟
المستقبلات تلعب دوراً رئيسياً في التعرف على الميكروبات وتفعيل الاستجابات المناعية.
- الخطوة 4: اختيار الإجابة الصحيحة
الخيار الأقرب لوظيفة كرات الدم البيضاء هو المستقبلات.

الإجابة النهائية:
ب) مستقبلات


مثال 3:
السؤال:
ماذا يسمى العدد 50 في عملية الطرح : 70 – 20 = 50 ؟

الخيارات:
أ) ناتج الطرح
ب) المطروح
ج) المطروح منه
د) لاشيء مما سبق

التحليل:
- الخطوة 1: فهم السؤال
السؤال هو: بماذا يسمى العدد 50 في عملية الطرح 70 – 20 = 50 ؟
في عملية الطرح:
العدد الأول (70) يسمى المطروح منه.
العدد الثاني (20) يسمى المطروح.
الناتج (50) يسمى ناتج الطرح أو الفرق.
- الخطوة 2: تحليل الخيارات
ناتج الطرح: هو العدد الناتج من عملية الطرح — وهو 50 في هذه الحالة.
المطروح: هو العدد الذي يتم طرحه — أي 20، وليس 50.
المطروح منه: هو العدد الذي نطرح منه — أي 70، وليس 50.
لاشيء مما سبق: غير صحيح لأن الخيار (أ) صحيح.
- الخطوة 3: تقييم الخيارات
العدد 50 هو ناتج عملية الطرح.
- الخطوة 4: اختيار الإجابة الصحيحة
الإجابة الصحيحة هي ناتج الطرح.

الإجابة النهائية:
أ) ناتج الطرح

السؤال الذي يجب عليك الإجابة عليه:
السؤال:
'''

CoT = []
for i, ques in enumerate(math['Question']):
    CoT.append(prompt_cot + ques + '\n الخيارات:\nأ) ' + str(math['Option 1'].iloc[i]) + '\nب) ' + str(math['Option 2'].iloc[i]) + '\nج) ' + str(math['Option 3'].iloc[i]) + '\nد) ' + str(math['Option 4'].iloc[i]))

cot_math = pd.DataFrame()
cot_math['prompt'] = CoT
cot_math.to_excel('Math-QA-ArabicPrompt-CoT.xlsx', index = False)